# Building a Slack Bot for Your Scientific Agent

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

A web app, a notebook, or a CLI is the *wrong* place to deploy an LLM
assistant for a research group. Researchers don't open new tabs; they
already live in **Slack** (or Teams, or Discord, or whichever chat tool
your group uses). Meeting them there is the difference between *"that
cool demo nobody uses"* and *"the bot the group can't function without."*

This notebook walks through building a Slack bot powered by Claude that:
- listens to direct messages and channel @mentions,
- remembers per-user conversation context,
- calls real scientific tools when relevant (literature search via the
  OpenAlex API from notebook 08),
- replies with proper threading, chunking, and status reactions.

The reference architecture is taken from [the MIC Data Analysis FAQ
project](https://github.com/yijiang1/MIC_Data_Analysis_FAQ), the same
production system that fields questions for the X-ray group at APS.

## Why Slack (vs. anything else)

- **Researchers are already there.** Zero install. Mobile and desktop.
- **Group access for free.** Invite the bot to a channel; everyone uses it.
- **Threads, files, reactions** — built-in UX primitives that map well
  to agent workflows (status reactions, threaded reasoning).
- **Async by default.** Long-running tool calls are fine; users see a
  hourglass and come back later.
- **Audit trail.** The conversation history is a built-in log of what
  the agent did and was asked to do.

## What you'll see

Three runnable bots at increasing complexity, all in
`../slack_bot_examples/`:

1. `01_hello_world_bot.py` — minimal echo bot (~50 lines)
2. `02_claude_bot.py` — Claude + per-user memory + threading (~100 lines)
3. `03_tool_use_bot.py` — adds the OpenAlex literature-search tool (~180 lines)

Plus a step-by-step guide for setting up the Slack app itself in the
admin UI — the part that has no code but is where everyone gets stuck
the first time.

---
## Architecture: Socket Mode vs. webhook

Slack offers two ways to receive events:

| Mode | What you need | When to use |
|---|---|---|
| **Webhook** (HTTP) | A public HTTPS URL Slack can POST to | Production deployments behind a load balancer |
| **Socket Mode** (WebSocket) | Just a network connection out | Development, prototypes, anything behind a firewall |

We use **Socket Mode** for this tutorial: it works from any laptop with
internet, no port forwarding, no ngrok. Same code can switch to webhook
mode later by swapping the handler.

The library is `slack-bolt` — Slack's official Python SDK, async-friendly,
and handles all the WebSocket plumbing.

---
## Setup, in 5 steps

### 1. Create the Slack app

Go to <https://api.slack.com/apps> → **Create New App** → **From scratch**.
Name it (e.g. *MRS Tutorial Bot*) and pick a workspace where you have
permission to install apps.

### 2. Enable Socket Mode + get the App-Level Token

In the new app's settings:
- **Socket Mode** (left sidebar) → toggle on.
- It'll prompt you to generate an **App-Level Token** with the
  `connections:write` scope. Generate it. The token starts with
  `xapp-...`. **This is your `SLACK_APP_TOKEN`**.

### 3. Add bot scopes

**OAuth & Permissions** → under *Bot Token Scopes*, add:

| Scope | Why |
|---|---|
| `app_mentions:read` | See `@yourbot` mentions in channels |
| `chat:write` | Post replies |
| `im:history` | Read messages users send the bot in DM |
| `im:read` | Know about DM channels |
| `im:write` | Send DMs |
| `reactions:write` | Add the hourglass status reaction (only needed for #3) |

### 4. Subscribe to events

**Event Subscriptions** → toggle on → under *Subscribe to bot events* add:
- `app_mention` — fires when the bot is `@`-tagged in a channel
- `message.im` — fires when the bot receives a DM

### 5. Install + grab the Bot Token

**Install App** (top of the OAuth page) → install to your workspace. Copy
the **Bot User OAuth Token**. Starts with `xoxb-...`. **This is your
`SLACK_BOT_TOKEN`**.

You'll always need both tokens: `xapp-` for the WebSocket connection and
`xoxb-` for actually calling Slack's API.

---
## Example 1 — Hello World

The smallest bot that demonstrates the slack-bolt + Socket Mode pattern.
~50 lines. Echoes your DM back, and replies to channel @mentions.

### Code

In [ ]:
"""Minimal Slack echo bot.

Run:
    export SLACK_BOT_TOKEN=xoxb-...
    export SLACK_APP_TOKEN=xapp-...
    python 01_hello_world_bot.py

What it does:
- Listens for direct messages and @mentions in any channel.
- Echoes the message text back, prefixed with 'You said: '.
- Stops cleanly on Ctrl-C.

Why slack-bolt + Socket Mode:
- No public URL or webhook server required (Socket Mode opens an outbound
  WebSocket to Slack instead of receiving inbound HTTPS).
- Works fine from a laptop behind a firewall, perfect for prototyping.
"""
import os
import asyncio
import logging

from slack_bolt.async_app import AsyncApp
from slack_bolt.adapter.socket_mode.aiohttp import AsyncSocketModeHandler

logging.basicConfig(level=logging.INFO)

# slack-bolt reads SLACK_BOT_TOKEN from the environment automatically.
app = AsyncApp(token=os.environ["SLACK_BOT_TOKEN"])


@app.event("message")
async def on_dm(event, say):
    """Triggered for every message the bot can see, including its own.

    `event["channel_type"] == "im"` filters to direct messages only.
    `event.get("subtype")` filters out bot messages, edits, joins, etc.
    """
    if event.get("channel_type") != "im":
        return
    if event.get("subtype"):                 # ignore edits, bot messages, etc.
        return
    await say(f"You said: {event['text']}")


@app.event("app_mention")
async def on_mention(event, say):
    """Triggered when the bot is @mentioned in a channel."""
    # event['text'] includes the mention itself, e.g. "<@U123456> hello"
    user_text = event["text"].split(">", 1)[1].strip()
    await say(f"<@{event['user']}> you said: {user_text}")


async def main():
    handler = AsyncSocketModeHandler(app, os.environ["SLACK_APP_TOKEN"])
    print("Bot is running. Send it a DM in Slack. Ctrl-C to stop.")
    await handler.start_async()


if __name__ == "__main__":
    asyncio.run(main())

### Run it

```bash
export SLACK_BOT_TOKEN=xoxb-...
export SLACK_APP_TOKEN=xapp-...
python slack_bot_examples/01_hello_world_bot.py
```

Then in Slack: find your bot under *Apps* in the left sidebar, click to
DM it, and say *"hello"*. It echoes back.

### What's happening
- `AsyncApp(token=...)` — the slack-bolt application; `token` is the
  `xoxb-` bot token.
- `@app.event("message")` — registers a handler. Slack-bolt does the
  routing for you.
- `event["channel_type"] == "im"` — filters to DMs (vs channel messages).
- `event.get("subtype")` — Slack uses subtypes for edits, joins, bot
  messages; we ignore those.
- `say(...)` — slack-bolt-injected helper that posts to the same channel
  as the triggering event.
- `AsyncSocketModeHandler(...).start_async()` — opens the WebSocket and
  blocks until you Ctrl-C.

---
## Example 2 — Claude with per-user conversation memory

Now the bot is actually useful. Same shape as #1, plus:
- An async Anthropic client.
- A `defaultdict(list)` keyed by Slack user ID for per-user chat history.
- A system prompt tuned for Slack (be concise; use Slack markdown).
- A `chunk()` helper for replies longer than Slack's 3000-char limit.
- Threaded replies — the bot answers in the same thread the user opened
  in (or starts one), so channels stay tidy.
- A `clear` / `reset` / `forget` command to wipe the user's history.

### Code

In [ ]:
"""Slack bot wired to Claude with per-user conversation memory.

Run:
    export SLACK_BOT_TOKEN=xoxb-...
    export SLACK_APP_TOKEN=xapp-...
    export ANTHROPIC_API_KEY=...
    python 02_claude_bot.py

What it does:
- Receives a DM or @mention.
- Looks up that user's prior chat history (in-process dict).
- Sends the full conversation to Claude with a system prompt.
- Posts the response back, chunked if Slack's 3000-char message limit is hit.
- Posts in the same thread as the user's message (if any) so threads stay tidy.

This is the smallest interesting "scientific Slack bot" — it remembers
context within a session and feels conversational. Production extras
(persistence across restarts, multi-user quotas, tool use) come next.
"""
import os
import asyncio
import logging
from collections import defaultdict

import anthropic
from slack_bolt.async_app import AsyncApp
from slack_bolt.adapter.socket_mode.aiohttp import AsyncSocketModeHandler

logging.basicConfig(level=logging.INFO)

MODEL = "claude-sonnet-4-6"
MAX_HISTORY_TURNS = 20      # cap on user+assistant messages kept per user
MAX_SLACK_CHARS = 3000      # Slack's per-message text limit (well under 4000)

SYSTEM_PROMPT = (
    "You are a helpful materials-science research assistant on Slack. "
    "Keep replies concise (under 200 words unless asked for more). "
    "Use Slack markdown: *bold*, _italic_, `code`, and ```code blocks```. "
    "Do not pretend to access tools you do not have."
)

app = AsyncApp(token=os.environ["SLACK_BOT_TOKEN"])
client = anthropic.AsyncAnthropic()

# Per-user in-process conversation store. Lost on restart — fine for a demo;
# in production back this with SQLite, Redis, etc.
HISTORY: dict[str, list[dict]] = defaultdict(list)


def chunk(text: str, limit: int = MAX_SLACK_CHARS) -> list[str]:
    """Split a long reply on paragraph boundaries to fit Slack's limit."""
    if len(text) <= limit:
        return [text]
    chunks, buf = [], ""
    for para in text.split("\n\n"):
        if len(buf) + len(para) + 2 > limit:
            chunks.append(buf.rstrip())
            buf = para + "\n\n"
        else:
            buf += para + "\n\n"
    if buf.strip():
        chunks.append(buf.rstrip())
    return chunks


async def claude_reply(user_id: str, user_text: str) -> str:
    """Append user_text to history, call Claude, append + return the reply."""
    history = HISTORY[user_id]
    history.append({"role": "user", "content": user_text})
    # Trim to last N turns to keep context small
    history[:] = history[-MAX_HISTORY_TURNS:]

    resp = await client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=history,
    )
    reply = "".join(b.text for b in resp.content if b.type == "text")
    history.append({"role": "assistant", "content": reply})
    return reply


async def handle(event, say):
    user_id = event["user"]
    text = event.get("text", "")
    # Strip the leading "<@BOTID>" if this came from an @mention
    if text.startswith("<@"):
        text = text.split(">", 1)[1].strip()

    if text.lower() in {"clear", "reset", "forget"}:
        HISTORY.pop(user_id, None)
        await say(text="🧹 Cleared our conversation history.",
                  thread_ts=event.get("thread_ts") or event.get("ts"))
        return

    try:
        reply = await claude_reply(user_id, text)
    except Exception as e:
        await say(text=f"⚠️ Error: `{e}`")
        return

    # Reply in-thread if the user opened one; otherwise top-level
    thread_ts = event.get("thread_ts") or event.get("ts")
    for piece in chunk(reply):
        await say(text=piece, thread_ts=thread_ts)


@app.event("message")
async def on_dm(event, say):
    if event.get("channel_type") != "im" or event.get("subtype"):
        return
    await handle(event, say)


@app.event("app_mention")
async def on_mention(event, say):
    await handle(event, say)


async def main():
    handler = AsyncSocketModeHandler(app, os.environ["SLACK_APP_TOKEN"])
    print(f"Bot is running with model {MODEL}. Type 'clear' to reset history. Ctrl-C to stop.")
    await handler.start_async()


if __name__ == "__main__":
    asyncio.run(main())

### Try it

```bash
export ANTHROPIC_API_KEY=sk-ant-...
python slack_bot_examples/02_claude_bot.py
```

Then DM the bot:

> *"Explain Bragg's law in two sentences."*

It replies. Then ask a follow-up that depends on context:

> *"Now do the same for Laue's formulation."*

It uses the prior turn — which is the whole point.

### Production-shaping notes
- **`HISTORY` is in-process** — restart the bot and it forgets everyone.
  In production back this with SQLite or Redis keyed by `(workspace_id, user_id)`.
- **Async client matters.** With sync `anthropic.Anthropic()`, every
  Claude call would block the bot's event loop and serialise all users.
  `AsyncAnthropic` lets multiple users get answered in parallel.
- **The `MAX_HISTORY_TURNS = 20` cap** keeps each call cheap. For long
  scientific conversations, consider summarising old turns instead of
  truncating them.

---
## Example 3 — Tool use (literature search)

Now the bot can *do things*, not just talk. We register one tool —
`search_recent_papers`, which hits the OpenAlex API from notebook 08 —
and Claude decides when to call it based on the user's question.

The added niceties on top of #2:
- A tool-use loop (same shape as notebook 04's from-scratch agent).
- An **hourglass reaction** ⏳ added to the user's message while the bot
  works, removed when done. Visible status without a separate "thinking..."
  message.

### Code

In [ ]:
"""Slack bot with Claude + tool use, talking to a real scientific tool.

Run:
    export SLACK_BOT_TOKEN=xoxb-...
    export SLACK_APP_TOKEN=xapp-...
    export ANTHROPIC_API_KEY=...
    python 03_tool_use_bot.py

What it does:
- Same per-user conversation memory + Slack threading as 02_claude_bot.
- Adds one tool: `search_recent_papers`, which hits the OpenAlex API
  (no key required) — the same tool used in tutorial notebook 08.
- Claude decides when to call it based on the user's question, runs
  the tool-use loop in the background, and posts a final answer.
- Posts a quick status reaction (hourglass) while working, then
  removes it when done — a nice UX touch on Slack.

This is roughly the smallest "scientific Slack assistant" you'd actually
ship to a research group: it answers questions, searches the literature
when relevant, and stays out of the way otherwise.
"""
import os
import json
import asyncio
import logging
from collections import defaultdict
from datetime import date, timedelta

import requests
import anthropic
from slack_bolt.async_app import AsyncApp
from slack_bolt.adapter.socket_mode.aiohttp import AsyncSocketModeHandler

logging.basicConfig(level=logging.INFO)

MODEL = "claude-sonnet-4-6"
MAX_TURNS = 6                # safety cap on tool-use loop iterations
MAX_HISTORY_TURNS = 20
MAX_SLACK_CHARS = 3000
MAILTO = os.environ.get("OPENALEX_MAILTO", "your.email@example.com")

SYSTEM_PROMPT = (
    "You are a materials-science research assistant on Slack. "
    "When asked about recent papers or literature, call the "
    "`search_recent_papers` tool rather than guessing. "
    "Otherwise answer concisely from your own knowledge. "
    "Use Slack markdown: *bold*, `code`. Keep replies under 200 words."
)

# ---- Tool definition (schema sent to Claude) ----
TOOLS = [{
    "name": "search_recent_papers",
    "description": "Search OpenAlex for recent scientific papers matching a query. "
                   "Returns title, authors, date, journal, and abstract.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query":       {"type": "string"},
            "since":       {"type": "string",
                            "description": "ISO date YYYY-MM-DD (default: 1 year ago)"},
            "max_results": {"type": "integer", "default": 5},
        },
        "required": ["query"],
    },
}]


def reconstruct_abstract(idx):
    if not idx:
        return ""
    pos = [(p, w) for w, ps in idx.items() for p in ps]
    pos.sort()
    return " ".join(w for _, w in pos)


def search_recent_papers(query: str, since: str = None,
                         max_results: int = 5) -> str:
    """Synchronous OpenAlex call. Returns JSON string for tool_result."""
    if since is None:
        since = (date.today() - timedelta(days=365)).isoformat()
    r = requests.get(
        "https://api.openalex.org/works",
        params={
            "search":  query,
            "filter":  f"from_publication_date:{since},type:article",
            "sort":    "publication_date:desc",
            "per_page": max_results,
            "mailto":  MAILTO,
        },
        timeout=30,
    )
    r.raise_for_status()
    out = []
    for w in r.json()["results"]:
        out.append({
            "title":   w.get("title", ""),
            "authors": ", ".join(a["author"]["display_name"]
                                 for a in w.get("authorships", [])[:5]),
            "date":    w.get("publication_date", ""),
            "journal": ((w.get("primary_location") or {}).get("source") or {})
                       .get("display_name", ""),
            "doi":     (w.get("doi") or "").replace("https://doi.org/", ""),
            "abstract": reconstruct_abstract(w.get("abstract_inverted_index"))[:600],
        })
    return json.dumps(out)


HANDLERS = {"search_recent_papers": search_recent_papers}

# ---- Bolt app + Claude client + per-user history ----
app = AsyncApp(token=os.environ["SLACK_BOT_TOKEN"])
client = anthropic.AsyncAnthropic()
HISTORY: dict[str, list[dict]] = defaultdict(list)


def chunk(text: str, limit: int = MAX_SLACK_CHARS) -> list[str]:
    if len(text) <= limit:
        return [text]
    chunks, buf = [], ""
    for para in text.split("\n\n"):
        if len(buf) + len(para) + 2 > limit:
            chunks.append(buf.rstrip()); buf = para + "\n\n"
        else:
            buf += para + "\n\n"
    if buf.strip():
        chunks.append(buf.rstrip())
    return chunks


async def claude_with_tools(user_id: str, user_text: str) -> str:
    """Run a tool-use loop until Claude is done, then return the final text."""
    history = HISTORY[user_id]
    history.append({"role": "user", "content": user_text})
    history[:] = history[-MAX_HISTORY_TURNS:]

    for _ in range(MAX_TURNS):
        resp = await client.messages.create(
            model=MODEL, max_tokens=1024,
            system=SYSTEM_PROMPT, tools=TOOLS,
            messages=history,
        )
        history.append({"role": "assistant", "content": resp.content})
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            return "".join(b.text for b in resp.content if b.type == "text")

        results = []
        for tu in tool_uses:
            try:
                payload = HANDLERS[tu.name](**tu.input)
            except Exception as e:
                payload = json.dumps({"error": str(e)})
            results.append({"type": "tool_result", "tool_use_id": tu.id,
                            "content": payload})
        history.append({"role": "user", "content": results})

    return "[ran out of turns — sorry]"


async def handle(event, say, client_):
    user_id = event["user"]
    channel = event["channel"]
    msg_ts  = event["ts"]
    thread_ts = event.get("thread_ts") or msg_ts

    text = event.get("text", "")
    if text.startswith("<@"):
        text = text.split(">", 1)[1].strip()
    if not text:
        return
    if text.lower() in {"clear", "reset", "forget"}:
        HISTORY.pop(user_id, None)
        await say(text="🧹 Cleared our conversation.", thread_ts=thread_ts)
        return

    # UX nicety: hourglass reaction while we work
    try:
        await client_.reactions_add(channel=channel, timestamp=msg_ts,
                                    name="hourglass_flowing_sand")
    except Exception:
        pass

    try:
        reply = await claude_with_tools(user_id, text)
        for piece in chunk(reply):
            await say(text=piece, thread_ts=thread_ts)
    except Exception as e:
        await say(text=f"⚠️ Error: `{e}`", thread_ts=thread_ts)
    finally:
        try:
            await client_.reactions_remove(channel=channel, timestamp=msg_ts,
                                           name="hourglass_flowing_sand")
        except Exception:
            pass


@app.event("message")
async def on_dm(event, say, client):
    if event.get("channel_type") != "im" or event.get("subtype"):
        return
    await handle(event, say, client)


@app.event("app_mention")
async def on_mention(event, say, client):
    await handle(event, say, client)


async def main():
    handler = AsyncSocketModeHandler(app, os.environ["SLACK_APP_TOKEN"])
    print(f"Tool-use bot running with model {MODEL}. Try: "
          "'find recent ptychography papers'. Ctrl-C to stop.")
    await handler.start_async()


if __name__ == "__main__":
    asyncio.run(main())

### Try it

```bash
export OPENALEX_MAILTO=your.email@inst.edu
python slack_bot_examples/03_tool_use_bot.py
```

Then DM:

> *"Find me three recent papers on X-ray ptychography and summarise them."*

You'll see:
1. ⏳ react to your message immediately.
2. After a few seconds, the bot replies with three real papers and DOIs.
3. The hourglass clears.

Try a non-search question:

> *"What is the difference between sol-gel and hydrothermal synthesis?"*

The bot answers from Claude's own knowledge, *without* calling the tool —
because the prompt doesn't match the tool's description. That's the
auto-routing behaviour from notebook 05's MCP discussion: the
`description:` field is the program.

---
## Patterns to take home

| Pattern | Lesson from |
|---|---|
| Socket Mode | No public URL needed; works behind firewalls |
| `@app.event("message")` + `channel_type == "im"` | Cleanly separate DM from channel handling |
| Threaded replies (`thread_ts=event.get("thread_ts") or event["ts"]`) | Keep channels tidy without losing context |
| Per-user `defaultdict` for history | Smallest viable conversation memory |
| `AsyncAnthropic` | Multi-user concurrency |
| `chunk()` on long replies | 3000-char message limit is real |
| Hourglass status reaction | Visible "I'm working" without a junk message |
| `clear` / `reset` keyword | Cheap escape hatch for stuck conversations |
| Slack markdown (`*bold*`, ``` ```code``` ```) in system prompt | Replies actually look good in Slack |

## What's missing (for production)

- **Persistence**: SQLite/Postgres-backed `HISTORY`, so restarts don't
  amnesia the bot.
- **Multi-workspace**: OAuth flow, per-workspace tokens stored encrypted.
- **Auth tiers**: e.g. only certain Slack users can call expensive tools.
- **File uploads**: `files_upload_v2` to send plots/PDFs back; handling
  user-uploaded files (the FAQ project does this for XRD/microscopy data).
- **Rate limits & retries**: respect Slack's per-second caps; back off
  on 429s. (`slack-bolt` does some of this for you, but custom retries on
  the LLM side are your responsibility.)
- **Health checks and metrics**: at minimum log every event timestamp +
  user ID + outcome to a structured store.
- **Graceful shutdown**: catch SIGTERM, drain in-flight conversations,
  close the socket cleanly.

The reference implementation at
<https://github.com/yijiang1/MIC_Data_Analysis_FAQ> (specifically
`services/slack_bot_agent.py`) shows what these look like in a system
that's actually running in production for a research group.

## Where to read more

- slack-bolt Python docs: <https://slack.dev/bolt-python/concepts>
- Slack API event reference: <https://api.slack.com/events>
- The reference codebase: <https://github.com/yijiang1/MIC_Data_Analysis_FAQ>